In [0]:
catalog = dbutils.widgets.get("catalog")
schema_prefix = dbutils.widgets.get("schema_prefix")

spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE SCHEMA {schema_prefix}_bronze")


In [0]:
from variables import folder_path

In [0]:
landing_folder_path = f"/Volumes/{catalog}/{schema_prefix}_staging/crm_files/landing/"

In [0]:
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{schema_prefix}_bronze.checkpoints")

In [0]:
from pyspark.sql.functions import current_timestamp, col

checkpoint_path = f"/Volumes/{catalog}/{schema_prefix}_bronze/checkpoints/crm_bronze"

(spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "csv")
  .option("cloudFiles.schemaLocation", checkpoint_path)
  .option("header", "true")
  .load(landing_folder_path)
  .withColumn("ingestion_timestamp", current_timestamp())
  .withColumn("source_file", col("_metadata.file_name"))
  .writeStream
  .option("checkpointLocation", checkpoint_path)
  .trigger(availableNow=True)
  .toTable("crm_bronze")
)

In [0]:
display(
    spark.read.table("crm_bronze")
)

In [0]:
%sql
SELECT * FROM cloud_files_state('/Volumes/tabfm_and_tsfm/marc_susagna_bronze/checkpoints/crm_bronze')

In [0]:
import shutil

source = f"/Volumes/{catalog}/{schema_prefix}_staging/crm_files/future/crm_leads02.csv"
dest = f"/Volumes/{catalog}/{schema_prefix}_staging/crm_files/landing/crm_leads02.csv"

shutil.copy(source, dest)
print(f"Copied {source} -> {dest}")

In [0]:
display(spark.read.option("versionAsOf", 1).table("crm_bronze"))

In [0]:
display(spark.sql("DESCRIBE HISTORY crm_bronze"))
